# 3B repair development — evidence, recovery and schema guard

**This is a repair-only development comparison, not the full Stage 2 evaluation.** It uses 17 newly constructed synthetic cases with fixed starting programs and fixture contracts. It does not load either intent parser, use an adapter, train anything, or read previous result ZIPs. References are used only for evaluation and the clearly labeled CPU infrastructure check; they are never supplied to the live model.

Four arms: **control** (original a1-v4), **evidence** (compact examples and one bounded fresh-code recovery), **schema_guard** (original agent plus declared output-column check), and **combined**. All use the same pinned **Qwen2.5-3B-Instruct in unquantized FP16**, three repair proposals and the existing token/tool limits. Recovery consumes the existing proposal budget. Successful starting programs are included as regression controls. The evidence arm also states the contract's rounding convention; its components are evaluated together in this first experiment.

Upload **stage2_repair_dev_source.zip** as one Kaggle dataset, then attach it here. Both intact ZIP and Kaggle's automatically extracted files work. Enable **Internet** and **T4 GPU** (one or two GPUs work). No 1.5B/3B adapter datasets are needed. Run all cells or Save Version → Run All. The CPU check runs first, then a two-case GPU smoke, then 17 × 4 = 68 paired evaluations in the full comparison.

Download `stage2_repair_dev_full.zip` and this executed notebook. The results are development diagnostics informed by previously seen failures, not an independent research benchmark. Local CPU checks cannot guarantee a particular Kaggle image or GPU service will work; the preflight checks the actual session before the full run.


In [1]:
import datetime,hashlib,json,os,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working')
ARMS=['control','evidence','schema_guard','combined']
RUN_DIR=WORK/('repair_dev_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
RUN_DIR.mkdir(parents=True)
print('Outputs:',RUN_DIR)


Outputs: /kaggle/working/repair_dev_20261115_055325_474625


In [2]:
MANIFEST_SHA='4be9feb5542ba53281961a772f97154d85836caab92af66f931d647d195521aa'
ROOT=WORK/('repair_dev_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source version'
    manifest=json.loads(raw)
    assert manifest['experiment']=='stage2-3b-repair-dev-v1'
    for name,expected in manifest['sha256'].items():
        relative=Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name)
        assert hashlib.sha256(data).hexdigest()==expected,f'Changed source: {name}'
        target=ROOT/relative;target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():assert target.read_bytes()==data,f'Changed working file: {name}'
        else:target.write_bytes(data)
    (ROOT/'_repair_dev_manifest.json').write_bytes(raw)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_repair_dev_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist())),'Duplicate archive entries'
            prefix='stage2_repair_dev_source/'
            if prefix+'manifest.json' not in z.namelist():continue
            raw=z.read(prefix+'manifest.json')
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach the stage2_repair_dev_source.zip dataset from the same folder as this notebook.'
PACKAGES=WORK/'repair_dev_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')
print('Verified source:',ROOT)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

Verified source: /kaggle/working/repair_dev_source_4be9feb5542b


In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(), 'Enable a GPU'; assert torch.cuda.get_device_properties(0).total_memory/2**30>=12; print(torch.__version__,torch.cuda.get_device_name(0))"],cwd=ROOT,env=ENV,check=True)


Removing optional torchao 0.10.0: incompatible with PEFT 0.21.2; this workflow does not use torchao.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
{"torchao": null, "action": "removed_incompatible", "previous_version": "0.10.0"}
{"probe": "cpu_lora_injection_and_forward", "trainable_parameters": 16, "status": "passed"}
2.11.0+cu130 Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch; assert torch.cuda.is_available(), 'Enable a GPU'; assert torch.cuda.get_device_properties(0).total_memory/2**30>=12; print(torch.__version__,torch.cuda.get_device_name(0))"], returncode=0)

In [4]:
def execute(name,extra):
    out=RUN_DIR/name;log=RUN_DIR/(name+'.log')
    command=[sys.executable,'-m','eval.stage2_repair_dev.runner','--out',str(out),'--packages',str(PACKAGES),'--arms',*ARMS,*extra]
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0,f'Job failed; inspect {log}. Download the log and partial output ZIP for diagnosis.'
    completed=json.loads((out/'completion.json').read_text())
    assert completed['complete'] and completed['records']==completed['expected']
    return out
CPU_CHECK=execute('stage2_repair_dev_cpu_check',['--cpu-check'])
assert not json.loads((CPU_CHECK/'completion.json').read_text())['model_quality_result']
print('CPU infrastructure checks passed. These scripted scores are not model results.')


1/68 dedup_original_first control: ACCEPT; correct=True
2/68 dedup_original_first evidence: ACCEPT; correct=True
3/68 dedup_original_first schema_guard: ACCEPT; correct=True
4/68 dedup_original_first combined: ACCEPT; correct=True
5/68 dedup_original_last evidence: ACCEPT; correct=True
6/68 dedup_original_last schema_guard: ACCEPT; correct=True
7/68 dedup_original_last combined: ACCEPT; correct=True
8/68 dedup_original_last control: ACCEPT; correct=True
9/68 dedup_complete_record schema_guard: ACCEPT; correct=True
10/68 dedup_complete_record combined: ACCEPT; correct=True
11/68 dedup_complete_record control: ACCEPT; correct=True
12/68 dedup_complete_record evidence: ACCEPT; correct=True
13/68 dedup_keep_repeated_groups combined: ACCEPT; correct=True
14/68 dedup_keep_repeated_groups control: ACCEPT; correct=True
15/68 dedup_keep_repeated_groups evidence: ACCEPT; correct=True
16/68 dedup_keep_repeated_groups schema_guard: ACCEPT; correct=True
17/68 dedup_crashed_candidate control: ACCEPT

In [5]:
SMOKE=execute('stage2_repair_dev_smoke',['--smoke'])
print('GPU smoke completed. A rejected task is not an infrastructure error. The next cell runs the full comparison.')


Fetching 2 files: 100%|██████████| 2/2 [00:35<00:00, 17.87s/it]

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 125.60it/s]
1/8 category_assignment_1 control: ESCALATE; correct=False
2/8 category_assignment_1 evidence: ACCEPT; correct=True
3/8 category_assignment_1 schema_guard: ESCALATE; correct=False
4/8 category_assignment_1 combined: ACCEPT; correct=True
5/8 filter_extra_helper evidence: ACCEPT; correct=False
6/8 filter_extra_helper schema_guard: ACCEPT; correct=True
7/8 filter_extra_helper combined: ACCEPT; correct=True
8/8 filter_extra_helper control: ACCEPT; correct=False
{
  "model_quality_result": true,
  "complete": true,
  "arms": {
    "control": {
      "tasks": 2,
      "correct": 0,
      "incorrect_accepted": 1,
      "withheld": 1,
      "initial_correct_lost": 0,
      "repairs": 0,
      "tokens": 3711,
      "llm_calls": 3,
      "tools": 0,
      "fresh_recovery_requests": 0,
      "diagnostic_seconds": 0,
      "repair_seconds": 10.084425833999944
    },

In [6]:
FULL=execute('stage2_repair_dev_full',[])
summary=json.loads((FULL/'summary.json').read_text())
assert summary['complete'] and summary['model_quality_result']
from IPython.display import display,FileLink
import pandas as pd
display(pd.DataFrame(summary['arms']).T)
display(FileLink(str(FULL.with_suffix('.zip'))))
print('Download the full ZIP and save/download this executed notebook. No separate collector is needed.')



Loading weights: 100%|██████████| 434/434 [00:03<00:00, 129.21it/s]
1/68 dedup_original_first control: ACCEPT; correct=True
2/68 dedup_original_first evidence: ACCEPT; correct=True
3/68 dedup_original_first schema_guard: ACCEPT; correct=True
4/68 dedup_original_first combined: ACCEPT; correct=True
5/68 dedup_original_last evidence: ACCEPT; correct=True
6/68 dedup_original_last schema_guard: ACCEPT; correct=True
7/68 dedup_original_last combined: ACCEPT; correct=True
8/68 dedup_original_last control: ACCEPT; correct=True
9/68 dedup_complete_record schema_guard: ACCEPT; correct=True
10/68 dedup_complete_record combined: ACCEPT; correct=True
11/68 dedup_complete_record control: ACCEPT; correct=True
12/68 dedup_complete_record evidence: ACCEPT; correct=True
13/68 dedup_keep_repeated_groups combined: ACCEPT; correct=True
14/68 dedup_keep_repeated_groups control: ACCEPT; correct=True
15/68 dedup_keep_repeated_groups evidence: ACCEPT; correct=True
16/68 dedup_keep_repeated_groups schema_guar

,tasks,correct,incorrect_accepted,withheld,initial_correct_lost,repairs,tokens,llm_calls,tools,fresh_recovery_requests,diagnostic_seconds,repair_seconds
control,17.0,11.0,1.0,5.0,0.0,6.0,39194.0,30.0,6.0,0.0,0.000000,85.057616
evidence,17.0,15.0,1.0,1.0,0.0,10.0,27434.0,17.0,0.0,4.0,0.048085,58.542871
schema_guard,17.0,12.0,0.0,5.0,0.0,7.0,40354.0,31.0,6.0,0.0,0.000000,86.401170
combined,17.0,16.0,0.0,1.0,0.0,11.0,28976.0,18.0,0.0,4.0,0.048771,63.138429


/kaggle/working/repair_dev_20261115_055325_474625/stage2_repair_dev_full.zip

Download the full ZIP and save/download this executed notebook. No separate collector is needed.
